# Interaction-profile collaborative filtering — results

**Compute runs externally** (server/GPU-optional); this notebook only loads the results and plots them.

Instead of borrowing a chemical neighbour's *embedding*, a molecule borrows its neighbours' **binding
profiles** (their rows of the train interaction matrix) over the ChemBERTa kNN graph, and the XGBoost
probe turns that into a prediction for the target protein. Parameter-free (numpy + boost). Bars:

- **raw** — `[mol | prot]`, no profile.
- **profile_A** — + the whole neighbour profile over all P proteins (`frac_pos`, `frac_neg`); the boost
  discovers protein–protein correlations itself (*fat* variant).
- **profile_B** — + neighbour profile read only at the target protein X, plus a version smoothed over X's
  protein neighbours ("protein Y correlated with X") — 4 scalars (*aligned* variant).
- **profile_A_own / profile_B_own** *(transductive only)* — also fold in the molecule's **own** train
  bindings to non-X proteins (target column masked). In inductive the query is cold, so these collapse
  to the neighbour-only bars and are omitted.

Both regimes, **5 repeats** each (transductive = LoRaX folds 1–5; inductive = cold-split seeds 42–46).
The inductive cold val/test split is **stratified to matched prevalence**. Embeddings are standardised.
Aggregation weights are **binary** (plain mean over k neighbours).

### Run the compute
```bash
bash scripts/modeling/eval/run_molecule_profile_cf.sh            # live dashboard
MAX_PARALLEL=4 bash scripts/modeling/eval/run_molecule_profile_cf.sh
```
Then run the cells below.

In [ ]:
import sys, pathlib, glob
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.stats import t as student_t

ROOT = pathlib.Path.cwd()
while not (ROOT / 'pyproject.toml').exists():
    if ROOT.parent == ROOT:
        raise RuntimeError('run from inside the orbind repository')
    ROOT = ROOT.parent

RESULT_DIR = ROOT / 'results/graph/mol_profile_cf'
COMBINED = RESULT_DIR / 'tables/molecule_profile_cf_runs.csv'
RUNS_DIR = RESULT_DIR / 'tables/runs'
FIGURES_DIR = ROOT / 'notebooks/figures'; FIGURES_DIR.mkdir(parents=True, exist_ok=True)
METRICS = ['AUROC', 'AUPRC', 'MCC', 'F1']
REGIMES = ['transductive', 'inductive_molecule']

# prefer the combined CSV; fall back to globbing per-unit CSVs (partial runs still render)
if COMBINED.exists():
    RES = pd.read_csv(COMBINED)
    print('loaded', COMBINED.relative_to(ROOT))
elif RUNS_DIR.exists() and glob.glob(str(RUNS_DIR / '*.csv')):
    RES = pd.concat([pd.read_csv(f) for f in sorted(glob.glob(str(RUNS_DIR / '*.csv')))], ignore_index=True)
    print(f'loaded {RES.repeat.nunique()} repeat-files from', RUNS_DIR.relative_to(ROOT))
else:
    RES = pd.DataFrame()
    print('No results yet. Run:  bash scripts/modeling/eval/run_molecule_profile_cf.sh')

def ci95_half(x):
    x = pd.Series(x).dropna().astype(float)
    if len(x) < 2:
        return np.nan
    return float(student_t.ppf(0.975, len(x) - 1) * x.std(ddof=1) / np.sqrt(len(x)))

if len(RES):
    n = RES.groupby('regime').repeat.nunique().to_dict()
    print('repeats per regime:', n)
    print('mean test prevalence:', RES.groupby('regime').test_prevalence.mean().round(3).to_dict())
    display(RES.groupby(['regime', 'bar'])[METRICS].mean().round(3))

In [ ]:
# ---- bar-grid: regimes (rows) x metrics (cols), mean +/- 95% t-CI over repeats ----
ORDER = ['raw', 'profile_A', 'profile_B', 'profile_A_own', 'profile_B_own']
LABEL = {'raw': 'raw', 'profile_A': 'profile\nA (fat)', 'profile_B': 'profile\nB (aligned)',
         'profile_A_own': 'A\n+own', 'profile_B_own': 'B\n+own'}
BAR_C = {'raw': '#9a9a95', 'profile_A': '#7fc7b8', 'profile_B': '#2f9e8f',
         'profile_A_own': '#8fb0e6', 'profile_B_own': '#2a78d6'}

if not len(RES):
    print('No results to plot yet.')
else:
    fig, axes = plt.subplots(len(REGIMES), len(METRICS), figsize=(15, 7), squeeze=False)
    for ri, regime in enumerate(REGIMES):
        d = RES[RES.regime == regime]
        bars = [b for b in ORDER if b in set(d.bar)]           # own-bars only present in transductive
        for mi, metric in enumerate(METRICS):
            ax = axes[ri, mi]
            means = [d[d.bar == b][metric].mean() for b in bars]
            errs = [ci95_half(d[d.bar == b][metric]) for b in bars]
            x = np.arange(len(bars))
            ax.bar(x, means, color=[BAR_C[b] for b in bars], edgecolor='white', linewidth=1.2,
                   yerr=np.nan_to_num(errs), error_kw=dict(ecolor='#33322f', capsize=3, lw=1), zorder=3)
            if means:
                ax.axhline(means[0], color=BAR_C['raw'], ls='--', lw=1, zorder=1)   # raw baseline
            ax.set_ylim(0, 1); ax.grid(alpha=.15, axis='y')
            ax.set_xticks(x); ax.set_xticklabels([LABEL[b] for b in bars], fontsize=8)
            for s in ('top', 'right'): ax.spines[s].set_visible(False)
            if ri == 0: ax.set_title(metric, fontsize=11)
            for xi, v in zip(x, means):
                if np.isfinite(v):
                    ax.text(xi, v + .02, f'{v:.3f}', ha='center', va='bottom', fontsize=8)
        nrep = d.repeat.nunique()
        axes[ri, 0].set_ylabel(f'{regime}\n({nrep} repeats)', fontsize=10)
    fig.suptitle('Interaction-profile CF — mean +/- 95% t-CI (stratified inductive split; '
                 'dashed = raw baseline)', fontsize=12.5, y=1.0)
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / 'molecule_profile_cf.png', dpi=150, bbox_inches='tight')
    plt.show()

## How to read

- **transductive** — molecules are warm, so `raw` already reads identity; the question is whether the
  neighbour/own binding profile adds *anything*. `profile_*_own` folds in the query's own non-X bindings.
- **inductive_molecule** — the real test. `profile_A`/`profile_B` **> raw** means a cold molecule can be
  scored from *how chemically-similar molecules bind*, without any interaction edge of its own. This is
  the collaborative-filtering cousin of the similarity-graph MP win.
- **A vs B** — if `profile_A` (fat, boost finds correlations) beats `profile_B` (aligned scalars), the
  cross-protein structure matters and is worth the extra columns; if they tie, the compact aligned form
  is enough and the explicit protein graph earns its keep.
- Compare on **AUROC** (prevalence-independent) primarily; the stratified split keeps val/test prevalence
  matched. Whiskers are 95% t-CI over 5 repeats.

Knobs via env: `K` (molecule neighbours), `KP` (protein neighbours), `MAX_PARALLEL`. Delete
`results/graph/mol_profile_cf/tables/` to regenerate.